In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import necessary libraries

In [ ]:
# importing
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Load Data and Models

In [ ]:
# load the data
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print(test.head())

# loading deberta model for multiple choice
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Setup predictions

In [ ]:
# the 5 options
options = ["A", "B", "C", "D", "E"]
# function to get prediction for one question
def get_prediction(row):
    prompt = str(row["prompt"])

    # make a (prompt, option) pair for each choice
    first = [prompt] * 5
    second = [str(row[o]) for o in options]

    # tokenize all 5 together
    inputs = tokenizer(
        first,
        second,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    )

    # reshape for multiple choice -> (batch=1, num_choices=5, seq_len)
    inputs = {k: v.unsqueeze(0).to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    logits = out.logits[0]  # 5 scores, one per option

    # rank options by score
    scored = list(zip(options, logits.tolist()))
    scored.sort(key=lambda x: x[1], reverse=True)

    top3 = [scored[0][0], scored[1][0], scored[2][0]]
    return " ".join(top3)

# Predictions

In [ ]:
preds = []
for i in range(len(test)):
    preds.append(get_prediction(test.iloc[i]))
    if i % 50 == 0:
        print("done", i)

# submission
submission = pd.DataFrame()
submission["id"] = test["id"]
submission["Prediction"] = preds
submission.to_csv("submission.csv", index=False)
print("saved!!")
submission.head()